# Quel modèle pour détecter les anomalies de salaire ?

On part de `employes.xlsx`, `bands.xlsx` et `market.xlsx`, et on lance plusieurs modèles sur les mêmes salariés.

1. Un seul tableau `base`, avec les ratios
2. Lancer les modèles
3. Qui est signalé ?
4. Les modèles signalent-ils les mêmes salariés ?
5. Le test : des anomalies qu'on connaît
6. Les salariés signalés par le plus de modèles

Chaque modèle donne un score à chaque salarié. On appelle **signalés** les salariés les mieux classés par un modèle (5 % du fichier, réglable).

Sous chaque graphe, un **Constat** donne les chiffres du fichier chargé. Il est recalculé à chaque lancement. Le texte au-dessus dit seulement comment lire.

Dans le même dossier que ce notebook : les trois fichiers Excel, `outils_anomalies.py` (les modèles et les graphes) et `outils_eda.py` (la lecture des fichiers). Si les Excel n'y sont pas, ce sont ceux de `input/` du projet qui sont lus ; la première cellule dit lesquels.

Une fois lancé sur les vraies données, ce fichier contient des salaires et des matricules : ne pas l'envoyer sur GitHub ou GitLab sans avoir effacé les sorties.

---
## 1. Un seul tableau

Doublons de matricule retirés, puis chaque salarié reçoit la grille et le marché de son métier et de son niveau.

| Colonne | Calcul | Lecture |
|---|---|---|
| `Compa_Ratio` | salaire / `Mid` | 0,90 = 10 % sous le milieu de grille |
| `Range_Penetration` | (salaire − `Min`) / (`Max` − `Min`) | 0 = au Min, 1 = au Max |
| `Market_Ratio` | salaire / `Median` | 1,00 = payé comme le marché |
| `Ratio_Grille_Marche` | `Mid` / `Median` | 1,10 = la grille paie 10 % de plus que le marché |
| `Rang_Performance` | la performance N-1 en rang | 1 = Très en dessous, 5 = Très au-delà |

Les modèles tournent sur `base` : les salariés qui ont un salaire et une grille.

In [ ]:
import outils_anomalies as outils

# 1. On lit employes.xlsx, bands.xlsx et market.xlsx dans le dossier du notebook, et on les croise :
# une ligne par salarié, avec Min, Mid, Max et Median de son métier + niveau
df = outils.lire_et_croiser()

# 2. On calcule les ratios du tableau ci-dessus, et le rang de la performance
df = outils.ajouter_ratios(df)
outils.constat_depart(df)

# 3. On garde les salariés qui ont un salaire et une grille : sans eux, pas de Compa-Ratio, et les modèles ne peuvent rien dire
base = df[(df['Fixe_Annuel_MAD'] > 0) & (df['Mid'] > 0)].copy()
print(f"Salariés donnés aux modèles : {len(base)} sur {len(df)} ({len(df) - len(base)} sans salaire ou sans grille).")
base.head()

---
## 2. Lancer les modèles

Deux familles, qui ne posent pas la même question.

**Ceux qui cherchent un profil rare.** Ils regardent toutes les colonnes de `COLONNES_PROFIL` d'un coup, et ne savent pas ce qu'est un salaire « bas » : un salarié très bien payé ou très âgé peut ressortir.

| Modèle | Ce qu'il fait |
|---|---|
| Isolation Forest | découpe les salariés au hasard : un profil rare se retrouve seul en peu de découpes. C'est l'IA d'origine du projet |
| Local Outlier Factor | compare chaque salarié à ses plus proches voisins |
| One-Class SVM | trace une frontière autour de la masse des salariés |
| Distance robuste | mesure la distance au profil « du milieu » |

**Ceux qui cherchent un salaire trop bas.** Ils comparent le salaire à une référence. Un salaire au-dessus de la référence a un score de 0.

| Modèle | La référence |
|---|---|
| Écart aux collègues | le Compa-Ratio des collègues de même métier et de même niveau |
| Salaire attendu, régression | le salaire prédit à partir de `COLONNES_ATTENDU`, par une addition d'effets |
| Salaire attendu, arbres | le même, par des arbres |

Pour retirer un modèle ou une colonne, effacer sa ligne.

In [ ]:
# ----- Réglages -----
PART_SIGNALEE = 0.05       # chaque modèle signale ses 5 % de salariés les mieux classés
TAILLE_GROUPE_MIN = 15     # « Écart aux collègues » : en dessous de 15 collègues, on compare à tout le niveau

MODELES = [
    'Isolation Forest',
    'Local Outlier Factor',
    'One-Class SVM',
    'Distance robuste',
    'Écart aux collègues',
    'Salaire attendu, régression',
    'Salaire attendu, arbres',
]

# Ce que regardent les modèles « profil rare » (les colonnes de l'Isolation Forest du projet)
COLONNES_PROFIL = ['Fixe_Annuel_MAD', 'Compa_Ratio', 'Market_Ratio', 'Age', 'Anciennete',
                   'Competence_N1', 'Rang_Performance', 'Hot_job']

# Ce que lisent les deux modèles « salaire attendu ». Le salaire n'y est pas : c'est ce qu'ils doivent deviner.
# « catégorie » : chaque valeur a son propre effet. « nombre » : un seul effet par unité en plus.
COLONNES_ATTENDU = {
    'Job_Family': 'catégorie',           # le métier
    'Niveau': 'catégorie',               # le niveau
    'Mid': 'nombre',                     # le milieu de la grille
    'Ratio_Grille_Marche': 'nombre',     # Mid / médiane du marché
    'Anciennete': 'nombre',              # en années
    'Age': 'nombre',                     # en années
    'Competence_N1': 'nombre',           # la note de compétence de l'an dernier
    'Performance_N1': 'catégorie',       # la performance de l'an dernier
    'Positionnement_9BOX': 'catégorie',  # la case 9-Box
    'Hot_job': 'nombre',                 # la tension du métier
    'Niveau_Etude_Max': 'catégorie',     # le niveau d'études
    'Cat_Ecole_Max': 'catégorie',        # la catégorie d'école
}

# scores : une ligne par salarié, une colonne par modèle. Plus le score est grand, plus le salarié ressort.
scores = outils.lancer(MODELES, base, COLONNES_PROFIL, COLONNES_ATTENDU, TAILLE_GROUPE_MIN)
# signes : pour chaque modèle, True pour ses salariés signalés
signes = outils.signales(scores, PART_SIGNALEE)

---
## 3. Qui est signalé ?

Pour chaque modèle, où sont ses signalés dans leur grille. La dernière ligne du tableau donne les mêmes chiffres pour tout le fichier : c'est le point de comparaison.

Un modèle utile pour la détection signale surtout des salariés bas dans leur grille. S'il signale beaucoup de salariés au-dessus du Max, il cherche autre chose : depuis le comité, un salaire trop haut n'est plus une anomalie.

Dans le graphe, une boîte par modèle : la moitié de ses signalés est dans la boîte, le trait du milieu est la médiane. Une boîte à gauche de la ligne noire : des signalés payés sous le milieu de leur grille.

In [ ]:
SEUIL_COMPA_BAS = 0.85     # le seuil de la règle « CompaRatio » de la détection

tableau = outils.tableau_signales(base, signes, SEUIL_COMPA_BAS)
display(tableau.round(2))
outils.graphe_signales(base, signes)
outils.constat_signales(tableau, SEUIL_COMPA_BAS)

---
## 4. Les modèles signalent-ils les mêmes salariés ?

Chaque case : la part des signalés du modèle de la ligne que le modèle de la colonne signale aussi. 100 % : les deux signalent les mêmes. 0 % : aucun en commun.

Deux modèles très d'accord font double emploi. Deux modèles en désaccord ne regardent pas la même chose : cela ne dit pas lequel a raison, c'est l'étape 5 qui le mesure.

In [ ]:
outils.graphe_accord(signes)

---
## 5. Le test : des anomalies qu'on connaît

Dans les vraies données, personne ne sait d'avance qui est une anomalie. On en fabrique donc : on baisse le salaire de quelques salariés tirés au hasard, on relance tous les modèles, et on compte combien de ces salariés chaque modèle retrouve parmi ses signalés.

Le fichier n'est pas touché : la baisse se fait sur une copie, en mémoire.

Le test est refait plusieurs fois avec d'autres salariés. Si deux modèles se croisent d'un essai à l'autre, on ne peut pas les départager.

Limite du test : il mesure un seul type d'anomalie, un salaire trop bas pour son poste et son profil. Il ne dit rien sur les erreurs de saisie ou les profils incohérents.

In [ ]:
# ----- Réglages -----
PART_FABRIQUEE = 0.03      # on baisse le salaire de 3 % des salariés
BAISSE = (0.12, 0.20)      # chacun perd entre 12 % et 20 % de son salaire
ESSAIS = 3                 # le test est refait 3 fois, avec d'autres salariés

# 1. Pour chaque essai : on copie base, on baisse le salaire des salariés tirés au hasard et on recalcule les ratios.
# 2. On relance tous les modèles sur cette copie, et chacun signale ses salariés les mieux classés.
# 3. On compte, pour chaque modèle, la part des salaires baissés qui sont parmi ses signalés.
resultats = outils.tester(MODELES, base, COLONNES_PROFIL, COLONNES_ATTENDU, TAILLE_GROUPE_MIN,
                          PART_FABRIQUEE, BAISSE, PART_SIGNALEE, ESSAIS)

# 4. On affiche le tableau (en %), du meilleur au moins bon, puis le graphe
display(resultats.round(1))
outils.graphe_test(resultats, PART_SIGNALEE)
outils.constat_test(resultats, PART_SIGNALEE)

Dans le graphe, la barre est la moyenne des essais, et le trait noir va du moins bon essai au meilleur. Le pointillé gris est ce que donnerait un tirage au hasard.

Pour un test plus dur, baisser moins les salaires (`BAISSE = (0.05, 0.10)`). Pour un test plus facile, les baisser plus.

---
## 6. Les salariés signalés par le plus de modèles

Sur les données d'origine (sans les salaires baissés du test). Un salarié signalé par plusieurs modèles qui ne regardent pas la même chose mérite d'être regardé en premier.

In [ ]:
outils.plus_signales(base, signes, combien=20)